# Plitt & Giocomo (2021) on dandiset 000054, part 2: population codes and inference

Paper. Plitt MH, Giocomo LM (2021). Experience-dependent contextual codes in the hippocampus.
*Nature Neuroscience* 24, 705-714.
[doi:10.1038/s41593-021-00816-6](https://doi.org/10.1038/s41593-021-00816-6). The claims file was
written from the NIH author manuscript in PubMed Central (PMC8893323) with its Methods, legends
and supplementary information.

Dandiset. [DANDI:000054](https://dandiarchive.org/dandiset/000054/0.260415.1712), version
0.260415.1712, the most recent published version; the paper cites only the draft, and the 2021
versions hold 10 of the 16 mice (see part 1, `01_sessions_place_cells.ipynb`, for the choice and
the data check of every file).

This notebook tests the population similarity, SF, Q estimate and model claims. The priors (M7)
need every trial a mouse ran before session 8; the dandiset holds only the imaged trials of the
imaging sessions (1, 3, 5 and 7 for most mice; none for R6), so the priors here are built from
those trials. This is a deviation for every claim that uses a posterior.

| Claim | Status |
|---|---|
# | fig3ab-block-versus-gradual-similarity | tested |
| edf7-checkerboard-sessions | tested |
| fig3f-zscored-difference-at-025 | tested |
| fig4-sf-stable-along-track | tested |
| fig6a-dkl-sign | tested (priors from imaged trials) |
| fig6bc-pooled-dkl | tested (priors from imaged trials) |
| fig6d-prior-reconstruction | tested (priors from imaged trials) |
| map-estimate-values | tested (priors from imaged trials) |
| edf9-early-sessions | tested (priors from imaged trials) |
| edf9k-null-models-rare | tested (priors from imaged trials) |
| edf9l-null-models-frequent | tested (priors from imaged trials) |
| edf10-decision-mice-like-frequent | tested (priors from imaged trials) |
| fig7-model-reproduces-remapping | tested (priors from imaged trials) |
| fig7g-model-prior-reconstruction | tested (priors from imaged trials) |
| edf3-simulated-posterior-population | tested (priors from imaged trials) |

The claims of part 1 and the two blocked claims are listed in the summary at the end.

In [ ]:
import time
import warnings
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import morph as M

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()

assets = M.asset_table()
print(f"dandiset {M.DANDISET_ID} version {M.DANDISET_VERSION}: {len(assets)} NWB files")
rf = assets.group != "decision"
pop = M.process_many([r for _, r in assets[rf & (assets.kind == "testing")].iterrows()], "population")
print(f"rare and frequent sessions 8 to N: {len(pop)} streamed in {time.time() - started:.0f} s")
sfs = M.process_many([r for _, r in assets[(~rf) | (rf & (assets.kind == "training") & (assets.day == 3))].iterrows()], "sf")
print(f"decision mice (all sessions) and session 3 of rare and frequent mice: {len(sfs)}; {time.time() - started:.0f} s")
meta = M.process_many([r for _, r in assets[rf & (assets.kind == "training") & (assets.day != 3)].iterrows()], "meta", workers=8)
print(f"other training files, trials only: {len(meta)}; total {time.time() - started:.0f} s")
RARE = ["R1", "R2", "R3", "R4", "R5", "R6"]
FREQ = ["F1", "F2", "F3", "F4", "F5", "F6"]
GROUP = {**{m: "rare" for m in RARE}, **{m: "frequent" for m in FREQ}}

## Data check

In [ ]:
allsess = pop + sfs + meta
chk = pd.DataFrame([{"mouse": s["mouse"], "group": s["group"], "kind": s["kind"], "day": s["day"], "part": s["part"],
                     "cells": s["n_cells"], "trials": len(s["trials"]), "frames": s["n_frames"],
                     "S0 trials": int((s["trials"].morph == 0).sum()), "S1 trials": int((s["trials"].morph == 1).sum()),
                     "starts_match": s["starts_match"], "stops_match": s["stops_match"],
                     "norm_check": s.get("norm_check"), "speed_kept": s.get("speed_kept"),
                     "empty_bins": s.get("empty_bins_a")} for s in allsess])
print("Files read, by group and kind:")
print(pd.crosstab(chk.group, chk.kind).to_string())
print("Trial tables match the flag series in every file:", bool(chk.starts_match.all() and chk.stops_match.all()))
print("Largest difference between the file's normalised series and activity / 99th percentile:",
      float(chk.norm_check.max()))
print("Fraction of frames at 2 cm/s or more, per session: "
      f"{chk.speed_kept.min():.3f} to {chk.speed_kept.max():.3f}; empty trial x bin cells after the speed "
      f"exclusion: up to {chk.empty_bins.max():.4f}")
print("\nSessions with fewer than two S-tilde = 0 or = 1 trials (no SF possible):")
print(chk[(chk["S0 trials"] < 2) | (chk["S1 trials"] < 2)][["mouse", "kind", "day", "trials", "S0 trials", "S1 trials"]].to_string(index=False))
tr = pd.concat([s["trials"].assign(mouse=s["mouse"], group=s["group"], day=s["day"]) for s in allsess])
print(f"\nS from {tr.S.min():.3f} to {tr.S.max():.3f}; f_h from {tr.fh.min():.2f} to {tr.fh.max():.2f}; "
      f"grid {M.unit_to_fh(M.GRID[0]):.2f} to {M.unit_to_fh(M.GRID[-1]):.2f} f_h, 1 unit = {M.UNIT_FH:.2f} f_h")
print("S-tilde = 0 and 1 trials map to units", M.wallmorphx(0.0).round(3), "and", M.wallmorphx(1.0).round(3))
sf_all = np.concatenate([s["sf"] for s in pop])
print(f"Whole-trial SF, sessions 8 to N: {np.nanmin(sf_all):.3f} to {np.nanmax(sf_all):.3f}, NaN {np.isnan(sf_all).sum()}")

Priors (M7): every trial each mouse ran before session 8 that the dandiset holds.

In [ ]:
before8 = tr[(tr.day < 8) & tr.group.isin(["rare", "frequent"])]
print("Trials before session 8 in the dandiset, per mouse (S-tilde counts):")
print(before8.groupby(["group", "mouse"]).morph.value_counts().unstack().fillna(0).astype(int).to_string())
mouse_prior = {m: M.prior_from_units(before8[before8.mouse == m].unit.to_numpy()) for m in before8.mouse.unique()}
print("Mice without trials before session 8:", sorted(set(RARE + FREQ) - set(mouse_prior)))
prior = {"rare": M.group_prior([mouse_prior[m] for m in RARE if m in mouse_prior]),
         "frequent": M.group_prior([mouse_prior[m] for m in FREQ if m in mouse_prior])}
# 2020 code variant: the group prior is the last mouse's prior (R6 has none, so R5)
prior_last = {"rare": mouse_prior["R5"], "frequent": mouse_prior["F6"]}
POST = {(lik, g): M.posterior(prior[g], lik) for lik in M.LIKELIHOODS for g in ("rare", "frequent")}
POST_LAST = {g: M.posterior(prior_last[g]) for g in ("rare", "frequent")}
mass = lambda p: {f"{lo}-{hi}": round(float(p[(M.GRID >= lo) & (M.GRID < hi)].sum()), 3) for lo, hi in ((-0.3, 0.15), (0.15, 0.85), (0.85, 1.3))}
print("Prior mass by unit range (low extreme, intermediate, high extreme):")
print({g: mass(p) for g, p in prior.items()})
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(M.unit_to_fh(M.GRID), prior["rare"], label="rare"); ax[0].plot(M.unit_to_fh(M.GRID), prior["frequent"], label="frequent")
ax[0].set_xlabel("f_h"); ax[0].legend(); ax[0].set_title("group priors")
ax[1].imshow(POST[("methods var 8.23", "rare")], origin="lower", aspect="auto", extent=[M.GRID[0], M.GRID[-1]] * 2)
ax[1].set_title("rare posterior (rows fhat)")
plt.show()
print("prior ranges:", {g: (float(p.min()), float(p.max())) for g, p in prior.items()})

For the rare morph mice the trials the dandiset holds before session 8 include the imaged trials
of session 3, half of them intermediate, so the rare prior here has more intermediate mass than
one from every trial would (the paper reports fewer than 4% of trials intermediate). R6 has no
trials before session 8 in the dandiset and is left out of the rare prior.

In [ ]:
# Per-session Q estimates and Delta D_KL (M9 to M11), for the readings the claims use.
def session_q(sf, trials, anchors=M.MAP_ANCHORS_FH):
    t = M.transform_sf(sf, trials["morph"].to_numpy(), anchors)
    return M.q_unnorm(t, trials["unit"].to_numpy())


def can_sf(s):
    return (s["trials"].morph == 0).sum() >= 2 and (s["trials"].morph == 1).sum() >= 2


for s in pop + [x for x in sfs if can_sf(x)]:
    s["qu"] = session_q(s["sf"], s["trials"])
    s["q"] = M.normalise_rows(s["qu"])
    s["dkl"] = {lik: M.delta_dkl(s["q"], POST[(lik, "rare")], POST[(lik, "frequent")]) for lik in M.LIKELIHOODS}
    s["dkl_last"] = M.delta_dkl(s["q"], POST_LAST["rare"], POST_LAST["frequent"])
    if "sf_norm" in s:
        s["q_norm"] = M.normalise_rows(session_q(s["sf_norm"], s["trials"]))
        s["dkl_norm"] = M.delta_dkl(s["q_norm"], POST[("methods var 8.23", "rare")], POST[("methods var 8.23", "frequent")])
bad = sum(M.dkl_rows(POST[("methods var 8.23", g)][M.ROWS], s["q"][M.ROWS])[1] for s in pop for g in ("rare", "frequent"))
print("grid points left out for Q = 0 where P > 0:", bad)

## fig3ab-block-versus-gradual-similarity

> Single-session population trial-by-trial similarity matrices show two blocks in the rare morph
> condition and a gradual transition between the extremes in the frequent morph condition.

In [ ]:
Bm = {m: np.nanmean([s["C_raw"] for s in pop if s["mouse"] == m], axis=0) for m in RARE + FREQ}
share = {m: M.step_share(B) for m, B in Bm.items()}
for g, mice in (("rare", RARE), ("frequent", FREQ)):
    print(g, "step share per mouse:", {m: round(share[m], 3) for m in mice})
    print(f"  mean binned similarity, rows S-tilde = 0 and = 1:\n", np.round(np.mean([Bm[m] for m in mice], axis=0)[[0, 4]], 3))
obs, perms = M.mouse_permutation(share, RARE, FREQ)
print(f"rare mean {np.mean([share[m] for m in RARE]):.3f}, frequent {np.mean([share[m] for m in FREQ]):.3f}; "
      f"difference {obs:.3f}; 2.5 to 97.5 percentiles {np.percentile(perms, 2.5):.3f} to {np.percentile(perms, 97.5):.3f}; "
      f"outside: {bool(M.outside_95(obs, perms))}")
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, (m, d) in zip(ax, (("R2", 13), ("F4", 10))):
    s = [x for x in pop if x["mouse"] == m and x["day"] == d][0]
    a.imshow(s["C_full_sorted"], cmap="magma"); a.set_title(f"{m} session {d}, trials sorted by S")
plt.show()
print("similarity range in the two panels:", [(float(np.min(x["C_full_sorted"])), float(np.max(x["C_full_sorted"])))
                                              for x in pop if (x["mouse"], x["day"]) in (("R2", 13), ("F4", 10))])

**Result: not reproduced.** The step share (the largest fall between neighbouring S-tilde values
over the whole fall, mean of the S-tilde = 0 and = 1 rows of each mouse's binned matrix) is 0.605
for rare morph mice (0.485 to 0.792) and 0.563 for frequent morph mice (0.434 to 0.749); the
difference, 0.042, lies inside the 2.5 to 97.5 percentiles of the mouse permutations (-0.116 to
0.116). The rare morph rows fall mostly between S-tilde 0.25 and 0.5 (0.439 to 0.339) and the
frequent morph rows fall more evenly, but the measure the criterion names does not separate the
groups.

## edf7-checkerboard-sessions

> In six named rare morph sessions, trials near S-tilde = 0.25 randomly take one extreme
> representation or the other, seen as checkerboard bands in the sorted similarity matrices.

In [ ]:
named = [("R1", 8), ("R2", 13), ("R3", 9), ("R4", 11), ("R5", 8), ("R6", 10)]
ok_all = True
for m, d in named:
    s = [x for x in pop if x["mouse"] == m and x["day"] == d][0]
    sf = s["sf"][s["trials"].morph == 0.25]
    ok = (sf > 0.5).any() and (sf < 0.5).any()
    ok_all &= ok
    print(f"{m} session {d}: {len(sf)} S-tilde = 0.25 trials, SF above 0.5: {(sf > 0.5).sum()}, below: {(sf < 0.5).sum()}, "
          f"range {sf.min():.3f} to {sf.max():.3f}")
print("criterion met in all six:", ok_all)
others = [x for x in pop if x["group"] == "rare" and (x["mouse"], x["day"]) not in named]
print("other rare sessions with S-tilde = 0.25 SF on both sides of 0.5:",
      sum(((x["sf"][x["trials"].morph == 0.25] > 0.5).any() and (x["sf"][x["trials"].morph == 0.25] < 0.5).any()) for x in others),
      "of", len(others))

**Result: partially reproduced.** In five of the six named sessions the S-tilde = 0.25 trials
include SF values on both sides of 0.5 (R1 session 8: 5 above and 15 below; R2 session 13: 3 and
17; R3 session 9: 1 and 19; R4 session 11: 1 and 15; R6 session 10: 4 and 5). In R5 session 8
all 16 lie below 0.5 (0.424 to 0.465). Most other rare sessions (18 of 20) also have S-tilde =
0.25 trials on both sides.

## fig3f-zscored-difference-at-025

> After z-scoring within session, trial-by-trial population similarity differs significantly
> between rare and frequent morph mice for S-tilde = 0.25 trials.

In [ ]:
def zdiff(key, mouse_z=False, rare=RARE, freq=FREQ):
    if mouse_z:
        per = {m: M.zscore_matrix(np.nanmean([s[key] for s in pop if s["mouse"] == m], axis=0)) for m in rare + freq}
    else:
        per = {m: np.nanmean([M.zscore_matrix(s[key]) for s in pop if s["mouse"] == m], axis=0) for m in rare + freq}
    return M.mouse_permutation(per, rare, freq)


fig3f = {}
for label, key, mz in (("main: smoothed deconvolved, z-scored per session", "C_raw", False),
                       ("variant: 99th-percentile normalised signal", "C_norm", False),
                       ("variant: unsmoothed (2020 code)", "C_unsmoothed", False),
                       ("variant: z-scored mouse means (2026 code)", "C_raw", True)):
    obs, perms = zdiff(key, mz)
    sig = M.outside_95(obs, perms)
    fig3f[label] = (obs, perms, sig)
    print(label)
    print(pd.DataFrame(obs, index=M.MORPHS, columns=M.MORPHS).round(3).to_string())
    lo, hi = np.percentile(perms, [2.5, 97.5], axis=0)
    print("  S-tilde = 0.25 row: difference", obs[1].round(3), "\n  2.5 pct", lo[1].round(3), "\n  97.5 pct", hi[1].round(3))
    print("  significant entries of the 0.25 row:", [(M.MORPHS[j], round(obs[1, j], 3)) for j in range(5) if sig[1, j]])
    print("  all significant entries:", [(M.MORPHS[i], M.MORPHS[j], round(obs[i, j], 3)) for i, j in zip(*np.nonzero(sig)) if i <= j])

**Result: reproduced.** In the S-tilde = 0.25 row of the z-scored rare minus frequent
difference, the entries with S-tilde = 0.5 (-0.409) and 1.0 (-0.367) lie outside the 2.5 to 97.5
percentiles of the 923 mouse permutations: rare morph 0.25 trials are less similar to 0.5 and 1.0
trials than frequent morph ones. The entry with S-tilde = 0 is +0.281 against a 97.5 percentile of
0.290. The variants (normalised signal, no smoothing, z-scoring the mouse means) also find
significant entries in the 0.25 row.

## fig4-sf-stable-along-track

> Position-wise SF shows stable clustered representations along the whole track in rare morph
> mice and a spectrum in frequent morph mice; remapping happens soon after the trial starts and
> stays as the mouse runs.

In [ ]:
curves = {}
for m in RARE + FREQ:
    per_sess = []
    for s in pop:
        if s["mouse"] != m:
            continue
        mor = s["trials"].morph.to_numpy()
        per_sess.append(np.array([np.nanmean(s["sf_pos"][mor == v], axis=0) for v in M.MORPHS]))
    curves[m] = np.nanmean(per_sess, axis=0)            # 5 x bins
res4 = {}
for g, mice in (("rare", RARE), ("frequent", FREQ)):
    c = np.array([curves[m] for m in mice])
    mean, sem = c.mean(0), c.std(0, ddof=1) / np.sqrt(len(mice))
    res4[g] = (mean, sem)
    sep = np.abs(mean[0] - mean[4]) > sem[0] + sem[4]
    print(f"{g}: S-tilde 0 and 1 separated by more than the summed s.e.m. in {sep[1:].sum()} of 44 bins after the first")
    print("  mean SF by S-tilde, every 5th bin:\n", np.round(mean[:, ::5], 3))
mean, sem = res4["rare"]
ov = np.abs(mean[3] - mean[4]) <= sem[3] + sem[4]
near0 = np.abs(mean[1] - mean[0]) < np.abs(mean[1] - mean[4])
print(f"rare: 0.75 and 1.0 overlap within s.e.m. in {ov.sum()} of 45 bins; 0.25 nearer 0 than 1 in {near0.sum()} of 45 "
      f"bins ({near0[1:].sum()} of 44 after the first)")
mean, sem = res4["frequent"]
between = (np.minimum(mean[0], mean[2]) < mean[1]) & (mean[1] < np.maximum(mean[0], mean[2]))
print(f"frequent: 0.25 between 0 and 0.5 in {between.sum()} of 45 bins")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for a, g in zip(ax, ("rare", "frequent")):
    mean, sem = res4[g]
    for k, v in enumerate(M.MORPHS):
        a.plot(np.arange(45) * 10 + 5, mean[k], color=plt.cm.cool(v), label=str(v))
        a.fill_between(np.arange(45) * 10 + 5, mean[k] - sem[k], mean[k] + sem[k], color=plt.cm.cool(v), alpha=0.3)
    a.set_title(g); a.set_xlabel("position (cm)")
ax[0].set_ylabel("SF"); ax[0].legend()
plt.show()
print("plotted range:", {g: (float((res4[g][0] - res4[g][1]).min()), float((res4[g][0] + res4[g][1]).max())) for g in res4})

**Result: reproduced.** In both conditions the S-tilde = 0 and = 1 curves of position-wise SF are
separated by more than their summed s.e.m. in all 44 bins after the first. In rare morph mice the
S-tilde 0.75 and 1.0 curves overlap within s.e.m. in all 45 bins and the 0.25 curve lies nearer the
S-tilde 0 curve in all 45. In frequent morph mice the 0.25 curve lies between the 0 and 0.5 curves
in all 45 bins. The curves are flat along the track (printed every 5th bin).

## fig6a-dkl-sign

> Delta D_KL of each session's Q estimate took the expected sign (negative for rare, positive for
> frequent) in 41 of 45 sessions and in every session of 9 of 12 mice; across mice, rare -0.88 +-
> 0.23 and frequent 0.50 +- 0.11 (mean +- s.e.m.), a significant difference.

In [ ]:
def fig6a_summary(values, label):
    d = pd.DataFrame([{"mouse": s["mouse"], "group": s["group"], "day": s["day"], "dkl": v} for s, v in zip(pop, values)])
    d["expected"] = np.where(d.group == "rare", d.dkl < 0, d.dkl > 0)
    per = d.groupby("mouse").dkl.mean()
    every = d.groupby("mouse").expected.all()
    obs, perms = M.mouse_permutation(per.to_dict(), RARE, FREQ)
    r, f = per[RARE], per[FREQ]
    out = {"reading": label, "sessions expected sign": f"{int(d.expected.sum())} of {len(d)}",
           "mice every session": int(every.sum()), "rare mean": round(r.mean(), 3), "rare sem": round(r.sem(), 3),
           "frequent mean": round(f.mean(), 3), "frequent sem": round(f.sem(), 3), "difference": round(obs, 3),
           "perm range": (round(perms.min(), 3), round(perms.max(), 3)), "outside range": bool(M.outside_range(obs, perms))}
    return out, d


rows = []
main6a, d6a = fig6a_summary([s["dkl"]["methods var 8.23"] for s in pop], "main: Methods likelihood")
rows.append(main6a)
for lik in ("code 0.3 units", "code 0.4 units"):
    rows.append(fig6a_summary([s["dkl"][lik] for s in pop], f"variant: likelihood {lik}")[0])
rows.append(fig6a_summary([s["dkl_last"] for s in pop], "variant: last mouse's prior (2020 code)")[0])
rows.append(fig6a_summary([s["dkl_norm"] for s in pop], "variant: normalised signal")[0])
print(pd.DataFrame(rows).to_string(index=False))
print("\nExpected: 41 of 45 sessions (+-4.1), 9 mice (+-0.9), rare -0.88 +- 0.23, frequent 0.50 +- 0.11")
print("\nPer session, main reading:")
print(d6a.pivot_table(index="mouse", columns="day", values="dkl").round(2).to_string())
m = d6a.groupby("mouse").dkl.mean()
print("per mouse:", m.round(3).to_dict())
print(f"\nwithin tolerance: sessions {abs(int(d6a.expected.sum()) - 41) <= 4.1}, mice {abs(main6a['mice every session'] - 9) <= 0.9}, "
      f"rare {abs(main6a['rare mean'] + 0.88) <= 0.23}, frequent {abs(main6a['frequent mean'] - 0.50) <= 0.11}")

**Result: partially reproduced.** With the likelihood of the Methods, Delta D_KL has the expected
sign in 37 of 47 sessions (41 +- 4.1 expected) and the difference between groups lies outside the
range of the 923 permutations (-0.361 against -0.321 to 0.361; the largest permutation is the
swapped grouping). The signs of the group means hold,
but the means are smaller than the paper's: rare -0.354 +- 0.046 (expected -0.88 +- 0.23) and
frequent 0.007 +- 0.025 (0.50 +- 0.11), and 7 mice have the expected sign in every session (9
expected). With the code's likelihood of 0.4 units, a variant, the values come closer (44 of 47
sessions, 9 mice, rare -0.841, frequent 0.219). The priors here are built from the imaged trials
only (see the header); the frequent morph values are near zero in most sessions.

## fig6bc-pooled-dkl

> Q estimates pooled over all sessions and mice approximate the posterior of the matching
> condition (Delta D_KL rare -0.44, frequent 0.42).

In [ ]:
def group_q(mice, reading, sess=pop):
    qs = []
    for m in mice:
        ss = [s for s in sess if s["mouse"] == m and "q" in s]
        if not ss:
            continue
        if reading == "a":
            qm = M.normalise_rows(np.mean([s["q"] for s in ss], axis=0))
        else:
            qm = M.normalise_rows(np.sum([s["qu"] for s in ss], axis=0))
        qs.append(qm)
    return M.normalise_rows(np.sum(qs, axis=0)), len(qs)


for reading in "ab":
    out = {}
    for g, mice in (("rare", RARE), ("frequent", FREQ)):
        q, n = group_q(mice, reading)
        out[g] = {lik: round(M.delta_dkl(q, POST[(lik, "rare")], POST[(lik, "frequent")]), 3) for lik in M.LIKELIHOODS}
    print(f"reading {reading} ({'mean of session Q' if reading == 'a' else 'one Q from all trials of a mouse'}):", out)
print("expected rare -0.44 +- 0.044, frequent 0.42 +- 0.042 (main likelihood decides)")

**Result: not reproduced.** Pooling the Q estimates gives rare -0.266 and frequent -0.000 when each
mouse's Q is the mean of its session Q estimates (reading a), and -0.268 and -0.004 when each
mouse's Q comes from all its trials (reading b), against -0.44 and 0.42. The rare value has the
expected sign; the frequent value does not, so neither the values nor the signs hold. With the
code's likelihood widths, variants, the frequent value is positive (0.116 to 0.162).

## fig6d-prior-reconstruction

> Priors reconstructed from each mouse's neural data alone are closer to the ideal prior of the
> mouse's own condition (Delta D_KL rare -0.46 +- 0.12, frequent 0.46 +- 0.07, mean +- s.e.m.
> across mice), a significant group difference.

In [ ]:
def prior_dkl(qu, lik="methods var 8.23", priors=prior):
    R = M.reconstruct_prior(qu, lik)
    return M.dkl(priors["rare"], R) - M.dkl(priors["frequent"], R), R


recon = {}
rows = []
for lik in M.LIKELIHOODS:
    vals = {}
    for m in RARE + FREQ:
        qu = np.sum([s["qu"] for s in pop if s["mouse"] == m], axis=0)
        vals[m], R = prior_dkl(qu, lik)
        if lik == "methods var 8.23":
            recon[m] = R
    obs, perms = M.mouse_permutation(vals, RARE, FREQ)
    r, f = pd.Series({m: vals[m] for m in RARE}), pd.Series({m: vals[m] for m in FREQ})
    rows.append({"likelihood": lik, "rare mean": round(r.mean(), 3), "rare sem": round(r.sem(), 3),
                 "frequent mean": round(f.mean(), 3), "frequent sem": round(f.sem(), 3), "difference": round(obs, 3),
                 "outside range": bool(M.outside_range(obs, perms)), "rare signs negative": int((r < 0).sum()),
                 "frequent signs positive": int((f > 0).sum())})
    if lik == "methods var 8.23":
        fig6d_vals = vals
        print("per mouse (main):", {k: round(v, 3) for k, v in vals.items()})
print(pd.DataFrame(rows).to_string(index=False))
print("expected rare -0.46 +- 0.12, frequent 0.46 +- 0.07")
fig, ax = plt.subplots(figsize=(6, 3))
for m in RARE + FREQ:
    ax.plot(M.unit_to_fh(M.GRID), recon[m], color="tab:blue" if m in RARE else "tab:orange", lw=0.8)
ax.plot(M.unit_to_fh(M.GRID), prior["rare"], "k--"); ax.plot(M.unit_to_fh(M.GRID), prior["frequent"], "k:")
ax.set_xlabel("f_h"); plt.show()
print("reconstructed prior range:", float(min(r.min() for r in recon.values())), float(max(r.max() for r in recon.values())))

**Result: partially reproduced.** The reconstructed priors separate the groups: the difference of
across-mouse means, -0.653, lies outside the range of all permutations, and all six frequent morph
mice have positive Delta D_KL (mean 0.633 +- 0.055, expected 0.46 +- 0.07). The rare morph mean is
-0.020 +- 0.089 (expected -0.46 +- 0.12), negative in 4 of 6 mice (R1 +0.178 and R2 +0.269). The
test and the signs of the means hold and both means are outside tolerance. The rare prior here is
built from imaged trials that include session 3, half of whose trials are intermediate.

## map-estimate-values

> The average MAP estimate of f_h for S-tilde = 0 trials is 11.08 under the rare and 10.64 under
> the frequent posterior, and for S-tilde = 1 trials 38.13 and 37.30.

In [ ]:
exp_map = {("rare", 0): 11.08, ("frequent", 0): 10.64, ("rare", 1): 38.13, ("frequent", 1): 37.30}
ttr = pd.concat([s["trials"].assign(group=s["group"]) for s in pop])
for lik in M.LIKELIHOODS:
    for reading in ("a: all sessions", "b: own group's sessions"):
        vals = {}
        for (g, v), e in exp_map.items():
            x = ttr[ttr.morph == v] if reading.startswith("a") else ttr[(ttr.morph == v) & (ttr.group == g)]
            vals[(g, v)] = float(np.mean(M.unit_to_fh(M.map_estimate(POST[(lik, g)], x.unit.to_numpy()))))
        ok = all(abs(vals[k] - e) <= 0.1 * e for k, e in exp_map.items())
        print(f"{lik:18s} {reading}: " + ", ".join(f"{g} S{v} {vals[(g, v)]:.2f}" for (g, v) in exp_map) + f"; all within 10%: {ok}")
vals_last = {(g, v): float(np.mean(M.unit_to_fh(M.map_estimate(POST_LAST[g], ttr[ttr.morph == v].unit.to_numpy())))) for (g, v) in exp_map}
print("variant last-mouse prior, reading a:", {f"{g} S{v}": round(x, 2) for (g, v), x in vals_last.items()})
print("f_h of the S-tilde = 0 and 1 trials themselves: means",
      round(float(ttr[ttr.morph == 0].fh.mean()), 2), round(float(ttr[ttr.morph == 1].fh.mean()), 2))

**Result: reproduced.** With the Methods likelihood the mean MAP estimates over all S-tilde = 0
and = 1 trials of sessions 8 to N are 10.67 and 10.72 (rare and frequent posterior) and 38.09 and
37.62, all within 10% of 11.08, 10.64, 38.13 and 37.30; each group's own trials (reading b) and
the code's likelihood widths give values within 10% as well.

## edf9-early-sessions

> The rare versus frequent remapping patterns were stable over sessions and emerged early in
> training (session 3) in most animals.

In [ ]:
s3 = [s for s in sfs if s["day"] == 3 and s["group"] in ("rare", "frequent") and "q" in s]
before3 = tr[(tr.day < 3) & tr.group.isin(["rare", "frequent"])]
mp3 = {m: M.prior_from_units(before3[before3.mouse == m].unit.to_numpy()) for m in before3.mouse.unique()}
prior3 = {"rare": M.group_prior([mp3[m] for m in RARE if m in mp3]), "frequent": M.group_prior([mp3[m] for m in FREQ if m in mp3])}
post3 = {g: M.posterior(prior3[g]) for g in prior3}
print("mice with trials before session 3 in the dandiset:", sorted(mp3))
rows = []
for s in s3:
    a = s["dkl"]["methods var 8.23"]
    b = M.delta_dkl(s["q"], post3["rare"], post3["frequent"])
    exp_sign = (lambda v: v < 0) if s["group"] == "rare" else (lambda v: v > 0)
    rows.append({"mouse": s["mouse"], "group": s["group"], "dkl a": round(a, 3), "dkl b": round(b, 3),
                 "expected a": exp_sign(a), "expected b": exp_sign(b)})
r9 = pd.DataFrame(rows)
print(r9.to_string(index=False))
print(f"session 3 with the expected sign: reading a {int(r9['expected a'].sum())} of {len(r9)}, reading b {int(r9['expected b'].sum())} of {len(r9)}")
cons = d6a.groupby("mouse").apply(lambda d: max((d.dkl < 0).mean(), (d.dkl > 0).mean()))
print("sessions 8 to N, share of each mouse's sessions with its majority sign:", cons.round(2).to_dict())
print("every mouse has one sign in most of its sessions:", bool((cons > 0.5).all()))

**Result: reproduced.** Session 3 Delta D_KL has the expected sign in 8 of 11 mice with the group
posteriors of sessions 8 to N (reading a) and in 8 of 11 with posteriors from the trials before
session 3 (reading b; R1 to R3 and F1 to F6 have such trials in the dandiset). In sessions 8 to N
every mouse's values have one sign in most of its sessions (60% to 100%).

## edf9k-null-models-rare and edf9l-null-models-frequent

> For rare (frequent) morph sessions, Q estimates are fit worse by the other condition's, uniform,
> Gaussian and multimodal prior posteriors and by a uniform posterior than by the matching
> posterior.

In [ ]:
unif = np.ones_like(M.GRID) / len(M.GRID)
gauss = np.exp(-((M.GRID - 0.5) / 0.2) ** 2); gauss /= gauss.sum()
multi = sum(w * np.exp(-((M.GRID - M.wallmorphx(sv)) / 0.08) ** 2) for w, sv in zip((1, 1, 10, 1, 1), M.MORPHS)); multi /= multi.sum()
nulls = {"uniform prior": M.posterior(unif), "Gaussian prior": M.posterior(gauss), "multimodal prior": M.posterior(multi),
         "uniform posterior": np.full((len(M.GRID), len(M.GRID)), 1 / len(M.GRID))}
for g, other in (("rare", "frequent"), ("frequent", "rare")):
    ss = [s for s in pop if s["group"] == g]
    own = np.array([M.mean_dkl(s["q"], POST[("methods var 8.23", g)]) for s in ss])
    print(f"=== {g} sessions ({len(ss)}); mean D_KL to own posterior {own.mean():.3f}")
    allok = True
    for name, P in {f"{other} posterior": POST[("methods var 8.23", other)], **nulls}.items():
        d = np.array([M.mean_dkl(s["q"], P) for s in ss])
        w = stats.wilcoxon(d - own)
        ok = w.pvalue < 0.05 and np.median(d - own) > 0
        allok &= ok
        print(f"  {name:20s}: mean D_KL {d.mean():.3f}, median null minus own {np.median(d - own):.3f}, "
              f"sessions with own closer {int((d > own).sum())} of {len(ss)}, Wilcoxon p = {w.pvalue:.2g}, holds {ok}")
    print(f"  all five hold: {allok}")

**Result: edf9k reproduced; edf9l partially reproduced.** For the 26 rare morph sessions every null
(frequent posterior, uniform, Gaussian and multimodal priors, uniform posterior) fits worse than
the rare posterior in all 26 sessions (Wilcoxon p = 3e-08 for each). For the 21 frequent morph
sessions the Gaussian prior, the multimodal prior and the uniform posterior fit worse than the
frequent posterior (p = 9.5e-07, 1.8e-05, 1.9e-06), but the rare posterior does not (11 of 21
sessions, p = 0.37) and the uniform prior fits better in 16 of 21 sessions (p = 0.096). Three of
the five frequent morph tests hold.

## edf10-decision-mice-like-frequent

> Frequent morph mice trained to report whether S was above or below 0.5 by where they licked
> (FD1 to FD4) remapped like frequent morph mice without the task.

In [ ]:
fdm = ["FD1", "FD2", "FD3", "FD4"]
fds = [s for s in sfs if s["group"] == "decision"]
print("decision sessions with an SF (S-tilde 0 and 1 trials present):", sum("q" in s for s in fds), "of", len(fds),
      "; without:", [(s["mouse"], s["day"]) for s in fds if "q" not in s])
for reading, sel in (("a: every imaged session", lambda s: True), ("b: sessions 8 to N", lambda s: s["day"] >= 8)):
    ss = [s for s in fds if "q" in s and sel(s)]
    q, n = group_q(fdm, "a", ss)
    pooled = {lik: round(M.delta_dkl(q, POST[(lik, "rare")], POST[(lik, "frequent")]), 3) for lik in M.LIKELIHOODS}
    recs = {}
    for m in fdm:
        mm = [s for s in ss if s["mouse"] == m]
        if mm:
            recs[m] = round(prior_dkl(np.sum([s["qu"] for s in mm], axis=0))[0], 3)
    per_sess = [round(s["dkl"]["methods var 8.23"], 2) for s in ss]
    print(f"{reading}: {n} mice, {len(ss)} sessions; pooled Q Delta D_KL {pooled}; reconstructed prior Delta D_KL per mouse {recs}")
    print(f"   per-session Delta D_KL (main): {per_sess}; positive in {sum(v > 0 for v in per_sess)} of {len(per_sess)}")

**Result: partially reproduced.** Reading (b) holds: each decision mouse's reconstructed prior is
closer to the frequent morph prior (Delta D_KL 0.494 to 0.601 over every imaged session, 0.520 to
0.635 over sessions 8 to N). Reading (a) does not: the Q estimate pooled over the decision mice
has Delta D_KL -0.042 (every imaged session) and -0.006 (sessions 8 to N) with the Methods
likelihood; it is positive only with the code's widths, variants (0.088 to 0.257). FD1's sessions
1 and 3 have no S-tilde = 0 or 1 trials and give no SF.

## fig7-model-reproduces-remapping and fig7g-model-prior-reconstruction

> A Hebbian network with radial basis inputs and k-winners-take-all competition, trained on
> stimuli drawn from the rare or frequent morph prior, gives Q estimates that match the matching
> posterior.
>
> Priors reconstructed from the model's Q estimates are closer to the prior of the matching
> condition, though less closely matched than for the neural data.

In [ ]:
FH_GRID = M.unit_to_fh(M.GRID)
MU_IN = np.linspace(FH_GRID[0], FH_GRID[-1], 100)


def hebbian_model(prior_g, rng, weight_noise, test_noise):
    """Methods, Hebbian learning model, with the claims file's parameters."""
    N, Mn, K = 100, 100, 20
    W = rng.uniform(0, 0.1, (N, Mn))
    stim = rng.choice(FH_GRID, size=1000, p=prior_g)

    def respond(f, noisy=True):
        x = np.exp(-((MU_IN[None, :] - (f + (2.74 * rng.standard_normal(np.shape(f)) if noisy else 0))[:, None]) ** 2) / 4.11)
        z = x @ W.T
        thr = -np.sort(-z, axis=1)[:, K - 1:K]
        y = np.maximum(0.01 * (z >= thr) * z + (0.01 * rng.standard_normal(z.shape) if noisy else 0), 0)
        return x, y

    for f in stim:
        x, y = respond(np.array([f]))
        dW = 0.1 * y.T @ x
        if weight_noise:
            dW = dW + 0.05 * rng.standard_normal(dW.shape)
        W = np.minimum(np.maximum(W + dW - 0.001, 0), 5.0)
    _, Y = respond(FH_GRID, noisy=test_noise)
    lo, hi = M.GRID < 0.05, M.GRID > 0.9
    c0, c1 = Y[lo].mean(0), Y[hi].mean(0)
    n = np.linalg.norm(Y, axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        g0 = Y @ c0 / (n * np.linalg.norm(c0)); g1 = Y @ c1 / (n * np.linalg.norm(c1))
        sf = g1 / (g0 + g1)
    a0, a1 = M.fh_to_unit(M.MAP_ANCHORS_FH[0]), M.fh_to_unit(M.MAP_ANCHORS_FH[1])
    m0 = np.nanmedian(sf[(M.GRID >= -0.1) & (M.GRID <= 0.1)]); m1 = np.nanmedian(sf[(M.GRID >= 0.9) & (M.GRID <= 1.1)])
    t = np.clip((sf - m0) / (m1 - m0) * (a1 - a0) + a0, M.GRID[0], M.GRID[-1])
    qu = M.q_unnorm(t, M.GRID.copy())
    return qu, int(np.isnan(sf).sum()), float(W.mean()), float(W.max())


t0 = time.time()
model = {}
for wn in (True, False):
    for tn in (True, False):
        for gi, g in enumerate(("rare", "frequent")):
            rng = np.random.default_rng(7000 + 100 * wn + 10 * tn + gi)
            model[(wn, tn, g)] = [hebbian_model(prior[g], rng, wn, tn) for _ in range(50)]
rows = []
for (wn, tn, g), runs in model.items():
    d = [M.delta_dkl(M.normalise_rows(r[0]), POST[("methods var 8.23", "rare")], POST[("methods var 8.23", "frequent")]) for r in runs]
    rows.append({"weight noise": wn, "test noise": tn, "trained on": g, "median Delta D_KL": round(float(np.median(d)), 3),
                 "IQR": (round(float(np.percentile(d, 25)), 3), round(float(np.percentile(d, 75)), 3)),
                 "models with NaN SF points": sum(r[1] > 0 for r in runs),
                 "mean weight after training": float(np.mean([r[2] for r in runs])), "largest weight": float(np.max([r[3] for r in runs]))})
fig7 = pd.DataFrame(rows)
print(fig7.to_string(index=False))
for wn in (True, False):
    for tn in (True, False):
        x = fig7[(fig7["weight noise"] == wn) & (fig7["test noise"] == tn)].set_index("trained on")["median Delta D_KL"]
        print(f"weight noise {wn}, test noise {tn}: rare median below 0 {x['rare'] < 0}, frequent above 0 {x['frequent'] > 0}")
print(f"model time {time.time() - t0:.0f} s")

In [ ]:
neural = {g: abs(np.mean([fig6d_vals[m] for m in mice])) for g, mice in (("rare", RARE), ("frequent", FREQ))}
print("neural across-mouse mean magnitude (fig6d, main):", {g: round(v, 3) for g, v in neural.items()})
for (wn, tn, g), runs in model.items():
    R = np.mean([M.reconstruct_prior(r[0]) for r in runs], axis=0)
    R = R / R.sum()
    v = M.dkl(prior["rare"], R) - M.dkl(prior["frequent"], R)
    ok_sign = v < 0 if g == "rare" else v > 0
    print(f"weight noise {wn}, test noise {tn}, trained on {g}: averaged reconstructed prior Delta D_KL {v:.3f} "
          f"(expected sign {ok_sign}; magnitude below neural {abs(v) < neural[g]})")

**Result: fig7 not reproduced; fig7g not reproduced.** Under no reading is the median Delta D_KL of
the rare-trained models below 0 while that of the frequent-trained models is above 0 (table). With
the parameters of the claims file and no weight noise, the weights decay from their initial mean
of 0.05 to a mean of 0.000004 during training, so the outputs carry no learned structure; with
weight noise of 0.05 per update the mean weight is 0.89 and the largest reaches the cap of 5, so
the weights follow the noise. The
averaged reconstructed prior has the expected sign for one condition only in each configuration
(rare-trained 0.155 to 0.301, frequent-trained 0.153 to 0.300 with test noise or weight noise). A
question on the input width and scale of the model has been asked.

## edf3-simulated-posterior-population

> A simulated population that stochastically encodes samples from the rare or frequent morph
> posterior reproduces the recorded remapping patterns.

In [ ]:
SIM_POST = {g: POST[("methods var 8.23", g)] for g in ("rare", "frequent")}
SIM_SESSIONS = []
for s_ in pop:
    rows_ = np.abs(M.GRID[None, :] - s_["trials"].unit.to_numpy()[:, None]).argmin(1)
    cdf_ = np.cumsum(SIM_POST[s_["group"]][rows_], axis=1)
    cdf_ /= cdf_[:, -1:]
    flat_ = (cdf_ + np.arange(len(rows_))[:, None]).ravel()        # rows stacked into one increasing array
    SIM_SESSIONS.append((s_["mouse"], s_["n_cells"], flat_, s_["trials"].morph.to_numpy()))


def _one_thread():
    from threadpoolctl import threadpool_limits
    threadpool_limits(1)


def simulate(args):
    """Sum over simulations of the z-scored binned similarity per mouse, for one reading.

    Reading a: one sample per trial and the session's cell count; b: 45 samples (one per position
    bin) and 100 neurons. Each trial's samples come from the group posterior at the trial's f_h by
    inverse transform sampling on the grid; neuron activity is gamma(kappa, 1)."""
    reading, n_sims, seed = args
    rng = np.random.default_rng(seed)
    acc, cnt = {}, {}
    for _ in range(n_sims):
        per_mouse = {}
        for mouse, ncells, flat, morph in SIM_SESSIONS:
            T = len(morph)
            n_samp, n_neur = (1, ncells) if reading == "a" else (45, 100)
            mu = np.linspace(FH_GRID[0], FH_GRID[-1], n_neur)
            u = rng.random((T, n_samp)) + np.arange(T)[:, None]
            k = np.searchsorted(flat, u) - np.arange(T)[:, None] * len(M.GRID)
            s_fh = FH_GRID[np.clip(k, 0, len(M.GRID) - 1)]
            kappa = 2 * np.exp(-((s_fh[:, :, None] - mu[None, None, :]) ** 2) / 4.11 ** 2)
            y = rng.gamma(np.maximum(kappa, 1e-300), 1.0).reshape(T, -1)
            yn = M.unit_rows(y)
            per_mouse.setdefault(mouse, []).append(M.zscore_matrix(M.binned_similarity(yn @ yn.T, morph)))
        for mouse, Bs in per_mouse.items():
            acc[mouse] = acc.get(mouse, 0) + np.mean(Bs, axis=0)
            cnt[mouse] = cnt.get(mouse, 0) + 1
    return acc, cnt


t0 = time.time()
sim = {}
for reading in "ab":
    jobs = [(reading, 250, 9000 + 10 * (reading == "b") + k) for k in range(4)]
    with ProcessPoolExecutor(4, initializer=_one_thread) as ex:
        parts = list(ex.map(simulate, jobs))
    per = {m: sum(p[0][m] for p in parts) / sum(p[1][m] for p in parts) for m in RARE + FREQ}
    diff = np.mean([per[m] for m in RARE], axis=0) - np.mean([per[m] for m in FREQ], axis=0)
    sim[reading] = diff
    print(f"reading {reading}: {sum(p[1]['R1'] for p in parts)} simulations; simulated rare minus frequent, z-scored:")
    print(pd.DataFrame(diff, index=M.MORPHS, columns=M.MORPHS).round(3).to_string())
obs, perms, sig = fig3f["main: smoothed deconvolved, z-scored per session"]
sig_entries = [j for j in range(5) if sig[1, j]]
print("recorded significant entries of the S-tilde = 0.25 row (fig3f):", [(M.MORPHS[j], round(obs[1, j], 3)) for j in sig_entries])
for reading in "ab":
    same = [bool(np.sign(sim[reading][1, j]) == np.sign(obs[1, j])) for j in sig_entries]
    print(f"reading {reading}: same sign at those entries {same}")
print(f"simulation time {time.time() - t0:.0f} s")

**Result: not reproduced.** The recorded S-tilde = 0.25 row has two significant entries
(fig3f-zscored-difference-at-025), with S-tilde 0.5 (-0.409) and 1.0 (-0.367). The simulated
z-scored rare minus frequent difference, averaged over 1,000 simulations, has the same sign at 0.5
(-0.145 with one sample per trial, -0.151 with 45 samples) but not at 1.0 (+0.009 under both
readings), so the criterion is not met.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Notebook | Outcome |
|---|---|---|
| cells-per-session | 1 | reproduced |
| session-and-trial-counts | 1 | partially reproduced |
| training-intermediate-fraction | 1 | blocked: data missing |
| place-cell-fractions | 1 | not reproduced |
| edf4a-fewer-place-cells-frequent | 1 | reproduced |
| edf4a-overlap-above-chance | 1 | partially reproduced |
| edf4bc-overlap-cells-at-ends-and-reward | 1 | not reproduced |
| fig2eh-place-cell-similarity-differs | 1 | partially reproduced |
| edf6-nmf-confirms-remapping | 1 | not reproduced |
| edf5-umap-rare-frequent-separable | 1 | reproduced |
| edf1-licking-before-reward | 1 | reproduced |
| fig3ab-block-versus-gradual-similarity | 2 | not reproduced |
| edf7-checkerboard-sessions | 2 | partially reproduced |
| fig3f-zscored-difference-at-025 | 2 | reproduced |
| fig4-sf-stable-along-track | 2 | reproduced |
| fig6a-dkl-sign | 2 | partially reproduced |
| fig6bc-pooled-dkl | 2 | not reproduced |
| fig6d-prior-reconstruction | 2 | partially reproduced |
| map-estimate-values | 2 | reproduced |
| edf9-early-sessions | 2 | reproduced |
| edf9k-null-models-rare | 2 | reproduced |
| edf9l-null-models-frequent | 2 | partially reproduced |
| edf10-decision-mice-like-frequent | 2 | partially reproduced |
| fig7-model-reproduces-remapping | 2 | not reproduced |
| fig7g-model-prior-reconstruction | 2 | not reproduced |
| edf3-simulated-posterior-population | 2 | not reproduced |
| edf2-fh-best-predictor | - | blocked: data missing |
| supp-earlier-morph-studies | - | blocked: other |

How the analysis differs from the paper's. The priors of every posterior-based claim come from
the imaged trials the dandiset holds before session 8, not from every trial; for the rare morph
mice these include the imaged trials of session 3, half of them intermediate, and R6 has none.
The dandiset holds 26 rare morph sessions 8 to N, two more than the paper's 24. Where the claims
file leaves a reading open, every reading it names is run (speed exclusion, denominators,
likelihood widths, shuffle counts, decision-mouse sessions). Choices not fixed by the claims file
are standard ones, named where they are made: smoothing of empty position bins by normalised
convolution, the 99.9th percentile of the shuffles for the UMAP filter, an NMF iteration limit of
2,000, the operational tests of "rises" (edf1) and of the peak shares (edf4bc), the tiling of the
model neurons over the stimulus grid (fig7, edf3), and testing the Hebbian model with and without
noise. Questions on the priors' trials, the Hebbian model's scale, the peak shares of edf4bc and
the simulated population have been asked (`questions/000054.yaml`).

What was found about the dataset. The version pinned holds all 16 mice; the 2021 versions hold
10. Every file holds only the imaged trials; the non-imaged trials and the sessions without imaging
(2, 4 and 6) are not in the dandiset, though the paper's priors and its training statistics need
them. The files hold imaging sessions 5 and 7 (and R4's session 6), which the paper does not
describe, and F6's session 5 has 4,103 cells, above the paper's range. FD3 has no session after 7,
so the decision-mouse counts cannot match the paper's. `session_start_time` is the date of
conversion (2026), not of the recording. FD1's sessions 1 and 3 hold continuous morph values. The
place cells stored in the JSON annotation (the authors' 2026 code) give fractions like this
attempt's and far above the paper's. No recording is deposited twice, and the activity shares the
clock of the position in every session.

In [ ]:
print(f"notebook run time {(time.time() - started) / 60:.1f} min")